# 注意力机制（Attention）

## 一、缩放点积注意力（Scaled Dot-Product Attention）

注意力机制要解决的问题是：**给定一个查询，如何从多个输入中找出与它相关的信息，并按相关程度加权融合？**

$$
\operatorname{Attention}(Q,K,V)
=\operatorname{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

- **Query（Q，查询向量）**：表示当前需要查找什么信息。
- **Key（K，键向量）**：用于和 Query 计算匹配分数。
- **Value（V，值向量）**：保存实际要参与加权融合的信息。
- **$d_k$**：每个 Query、Key 向量的维度。

Q、K、V 是向量或向量组成的矩阵，**不是词语本身**。它们通常由输入特征经过不同的可学习线性变换得到。Key 和 Value 可以来自同一批输入，但承担的作用不同。

### 1. $QK^\top$：为什么先计算匹配分数？

一个 Query 可能面对多个 Key。在融合信息前，需要判断它与每个 Key 有多匹配，否则就无法决定应当更关注哪一项信息。

对单个查询向量 $q$ 和键向量 $k$，使用点积得到一个分数：

$$
q\cdot k=\sum_{i=1}^{d_k}q_i k_i
$$

把所有 Query 与所有 Key 的点积放在一起，就得到分数矩阵 $QK^\top$。每个分数表示**当前 Query 与对应 Key 在模型所学特征空间中的匹配程度**，不一定等于人直观理解的语义相似度。

例如，某个 Query 对三个 Key 的缩放后分数为 $[1,3,2]$，说明第二个 Key 的分数最高。但这里还只是分数，尚未形成最终的注意力权重。

### 2. 除以 $\sqrt{d_k}$：为什么需要缩放？

向量维度较大时，点积的数值波动往往也会增大。如果把差距过大的分数直接送入 Softmax，输出可能非常接近 0 或 1，使梯度变小，影响训练。

可以用一个简化假设理解缩放系数的来源：假设 $q_i,k_i$ 独立，均值为 0、方差为 1，并且不同维度的乘积项之间也相互独立，则

$$
\begin{aligned}
\operatorname{Var}(q\cdot k)
&=\operatorname{Var}\left(\sum_{i=1}^{d_k}q_i k_i\right)\\
&=\sum_{i=1}^{d_k}\operatorname{Var}(q_i k_i)\\
&=d_k.
\end{aligned}
$$

因此，点积的标准差约为 $\sqrt{d_k}$。将其除以 $\sqrt{d_k}$：

$$
s=\frac{q\cdot k}{\sqrt{d_k}},\qquad
\operatorname{Var}(s)=\frac{d_k}{d_k}=1.
$$

这说明：**在上述假设下，缩放能抵消维度增大对分数波动范围的影响**，降低 Softmax 过度饱和的风险。实际网络中不保证方差严格等于 1。

### 3. Softmax：为什么要把分数转换成权重？

匹配分数可能为负，也没有固定的总和；如果直接拿来加权，融合结果会受到分数尺度的明显影响。Softmax 将同一个 Query 对应的分数转换成非负且总和为 1 的权重：

$$
a_i=\frac{e^{s_i}}{\sum_j e^{s_j}},\qquad \sum_i a_i=1.
$$

假设缩放后的分数为 $[1,3,2]$：

$$
\operatorname{Softmax}([1,3,2])\approx[0.090,0.665,0.245].
$$

| Key | 缩放后分数 | 注意力权重 |
|:---:|---:|---:|
| A | 1 | 9.0% |
| B | 3 | 66.5% |
| C | 2 | 24.5% |

注意：这里是 **Softmax 指数归一化**，不是简单除以分数总和，因此权重不是 $1/6、3/6、2/6$。如果有多个 Query，要分别对每个 Query 的分数进行 Softmax。

### 4. 乘以 $V$：为什么最后融合的是 Value？

前面的计算只回答了“应该关注哪些 Key、分别关注多少”。要真正获得信息，需要按照这些权重对对应的 Value 向量加权求和：

$$
z=\sum_i a_iV_i
$$

沿用上例：

$$
z=0.090V_A+0.665V_B+0.245V_C.
$$

Key 负责参与匹配，Value 负责提供融合的内容。最终得到的 $z$ 是一个**特征向量**，不是直接生成的词语、分类结果或推荐结果；它还可以送入后续网络进行处理。


## 二、交叉注意力（Cross-Attention）

### 场景：根据甲的观影历史推荐电影

已知甲的偏好信息，以及甲看过的 3 部电影的信息。现在希望从另外 5 部候选电影中推荐一部。

这里先通过交叉注意力融合甲的观影历史，再根据融合后的用户特征对候选电影评分。**交叉注意力本身负责信息融合，并不直接完成候选电影的排名。**

在这个例子中：

- **Q**：由甲的偏好向量生成，表示甲想看哪类电影。
- **K**：由 3 部已看电影的特征生成，用来与甲的 Q 计算匹配分数。
- **V**：同样由这 3 部已看电影的特征生成，提供电影信息用于加权融合。

Q 来自用户，K、V 来自电影，因此属于**交叉注意力**。这里说“甲作为 Query”“电影作为 Key/Value”只是简写，实际计算使用的是各自的向量表示。

### 1. 甲的 Q 与三部电影的 K 匹配

为什么先匹配？因为甲看过的三部电影，对描述甲当前兴趣的参考价值可能不同。先计算匹配分数，才能决定后面融合历史信息时各部电影的比重。

假设三部已看电影为 A、B、C，计算点积并经过 $\sqrt{d_k}$ 缩放后，得到：

$$
s=[1,3,2]
$$

这表示电影 B 与当前用户查询向量的匹配分数最高，其次是 C，最后是 A。这里只是人为设定的示例分数。

### 2. 对匹配分数进行 Softmax

为什么要做 Softmax？因为下一步需要用一组统一的权重来融合电影信息，而不是直接拿任意大小的分数相乘。

$$
a=\operatorname{Softmax}([1,3,2])\approx[0.090,0.665,0.245]
$$

| 已看电影 | 匹配分数 | 注意力权重 |
|:---:|---:|---:|
| A | 1 | 9.0% |
| B | 3 | 66.5% |
| C | 2 | 24.5% |

电影 B 的注意力权重最大，意味着在这次融合中它提供的信息占比较高。**注意力权重不是“甲喜欢这部电影的概率”。**

### 3. 按权重融合三部电影的 V

为什么加权的是 V，而不是 K？因为 K 已经用于计算匹配分数；V 才是提供给模型融合的电影信息。二者可以来自同一部电影，但通过不同的映射得到。

$$
z_{\text{甲}}=0.090V_A+0.665V_B+0.245V_C
$$

这样得到一个结合甲的偏好、并对已看电影信息进行加权聚合的特征向量。其中电影 B 的贡献相对更大，A 和 C 的信息也参与了融合。

### 4. 用融合后的用户特征匹配 5 部候选电影

为什么还有这一步？因为注意力输出 $z_{\text{甲}}$ 只是用户特征，并没有直接指出应该推荐哪部电影。

可以进一步将它与甲原有的用户特征结合，得到更新后的用户表示 $u'$，然后分别计算 5 部候选电影的推荐分数：

$$
r_j=\operatorname{Score}(u',c_j),\qquad j=1,\ldots,5.
$$

最后按分数排序，选择分数最高的电影。$\operatorname{Score}$ 的具体形式由推荐模型决定，可能是点积，也可能是一个神经网络；没有给出候选电影的特征，就不能确定实际推荐结果。

**交叉注意力的关键区别是：Q 和 K、V 来自不同的输入。** 在这个例子中，甲提供 Query，已看电影提供 Key 和 Value。


## 三、自注意力（Self-Attention）

### 场景：理解「苹果很甜」

假设句子「苹果很甜」被分成三个 Token：`[苹果, 很, 甜]`。这里是为说明过程而做的简化，实际分词结果取决于 Tokenizer。

单独看「苹果」，它可能指水果，也可能指品牌；看到「很甜」，我们更倾向于理解为水果。自注意力可以让一个 Token 结合其他 Token 的信息，形成新的上下文特征。

这里讨论**双向自注意力**：三个 Token 都可以读取整句话的信息，不遮挡后面的 Token。

自注意力中，Q、K、V 都由同一句话的输入特征生成：

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.
$$

因此，**每个 Token 都有各自的 Q、K、V 向量**。Q、K、V 来自同一组 Token，并不表示它们的数值一样。

### 1. 为什么要分别处理「苹果」「很」「甜」？

因为我们希望每个 Token 都获得一个结合上下文的新表示，而不是只得到整句话的一个向量。所以需要分别用三个 Token 的 Query 查询整句话的 Key，再加权融合 Value。

假设三个 Query 与三个 Key 计算得到的缩放后分数如下：

| 当前 Query | 苹果的 K | 很的 K | 甜的 K |
|:---:|---:|---:|---:|
| 苹果的 Q | 3 | 1 | 2 |
| 很的 Q | 1 | 3 | 2 |
| 甜的 Q | 2 | 1 | 3 |

表格中的每一行都属于一个 Query，**每一行分别做 Softmax**。这些分数只是用来说明计算过程，并非真实模型的输出。

### 2. 处理「苹果」

**为什么要参考其他 Token？** 只看「苹果」不容易确定它在句子中的具体含义，结合「很」「甜」提供的信息，有助于模型建立与水果、味道有关的上下文表示。

「苹果」的 Query 得到分数 $[3,1,2]$，通过 Softmax：

$$
a_{\text{苹果}}=\operatorname{Softmax}([3,1,2])
\approx[0.665,0.090,0.245]
$$

再用权重融合三个 Value：

$$
z_{\text{苹果}}=
0.665V_{\text{苹果}}+0.090V_{\text{很}}+0.245V_{\text{甜}}
$$

得到的不是新的文字「苹果」，而是**「苹果」这个 Token 更新后的特征向量**。它包含了其他 Token 的信息，有助于后续网络结合语境理解词义。注意力计算本身不保证直接完成消歧。

### 3. 处理「很」

**为什么也要读取上下文？** 「很」表示程度，但它修饰什么、在句子中起什么作用，需要结合周围的词来理解。

使用「很」的 Query，得到分数 $[1,3,2]$：

$$
a_{\text{很}}=\operatorname{Softmax}([1,3,2])
\approx[0.090,0.665,0.245]
$$

$$
z_{\text{很}}=
0.090V_{\text{苹果}}+0.665V_{\text{很}}+0.245V_{\text{甜}}
$$

此时「很」的特征融入了「苹果」「甜」的信息，为后续网络理解程度修饰关系提供依据。

### 4. 处理「甜」

**为什么「甜」也需要上下文？** 因为理解「甜」描述的对象，需要联系前面的「苹果」；理解修饰程度，还要联系「很」。

使用「甜」的 Query，得到分数 $[2,1,3]$：

$$
a_{\text{甜}}=\operatorname{Softmax}([2,1,3])
\approx[0.245,0.090,0.665]
$$

$$
z_{\text{甜}}=
0.245V_{\text{苹果}}+0.090V_{\text{很}}+0.665V_{\text{甜}}
$$

此时「甜」的特征同时考虑了自身以及前文提供的信息。

### 5. 三个 Token 都计算完成后，得到了什么？

三个 Token 分别得到一个新的特征向量：

$$
Z=\begin{bmatrix}
z_{\text{苹果}}\\
z_{\text{很}}\\
z_{\text{甜}}
\end{bmatrix}
=\operatorname{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.
$$

**为什么输入三个 Token，输出也是三个向量？** 因为每个 Token 都生成一个 Query，每个 Query 经过注意力计算后都会得到一个融合结果。输出向量的维度由 V 决定，不一定和输入向量维度完全相同。

这些特征可以再传给后续网络，用于机器翻译、文本分类、语义检索等任务。比如后续模型可以利用上下文信息，将「苹果」理解为水果，而不是手机品牌；但**自注意力层本身不直接输出翻译或分类结果**。

**自注意力的特点：Q、K、V 来自同一组 Token。** 这里使用双向自注意力，每个 Token 都能关注整句话。


## 四、因果自注意力（Causal Self-Attention）

### 场景：GPT 预测下一个 Token

继续使用 `[苹果, 很, 甜]` 作为例子。训练时，希望 GPT 学会：

- 已知 `苹果`，预测 `很`。
- 已知 `苹果 很`，预测 `甜`。

**为什么不能直接使用上面的双向自注意力？** 因为双向自注意力允许「苹果」读取后面的「很」。如果模型要用「苹果」预测「很」，就有可能提前读到答案。因此需要增加因果掩码（Causal Mask）。

因果自注意力仍然使用 Q、K、V 计算匹配分数并融合信息，**变化的是每个 Token 可以读取哪些 Token 的信息**。

### 1. 因果掩码：为什么不能读取后面的 Token？

| 当前 Query | 苹果的 K/V | 很的 K/V | 甜的 K/V |
|:---:|:---:|:---:|:---:|
| 苹果的 Q | ✓ | × | × |
| 很的 Q | ✓ | ✓ | × |
| 甜的 Q | ✓ | ✓ | ✓ |

例如：用「苹果」预测「很」时，只允许使用「苹果」的信息；用「苹果 很」预测「甜」时，允许使用「苹果」和「很」的信息。要预测的下一个 Token 不能参与当前计算。

在数学上，按 Token 在序列中的先后次序编号，令 $i$ 表示当前 Query 对应的序列索引，$j$ 表示 Key 对应的序列索引：

$$
M_{ij}=\begin{cases}
0, & j\le i,\\
-\infty, & j>i.
\end{cases}
$$

于是计算公式变成：

$$
\operatorname{CausalAttention}(Q,K,V)
=\operatorname{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.
$$

**为什么在 Softmax 之前加掩码？** 因为被遮挡的分数变成 $-\infty$，经过 Softmax 后相应权重为 0，最终不会读取这些 Token 的 Value。实现时可能使用足够小的数值或专用掩码运算。

### 2. 根据「苹果」预测「很」

为什么只融合「苹果」自身的信息？因为当前只能使用 `苹果` 这个前缀，「很」是下一步要预测的目标，不能提前参与计算。

沿用上一节的示例分数，「苹果」的 Query 对三个 Key 的缩放后分数为：

$$
[3,1,2]
$$

加入因果掩码后：

$$
[3,1,2]\longrightarrow[3,-\infty,-\infty]
$$

Softmax 得到：

$$
[1,0,0]
$$

所以注意力输出为：

$$
z_{\text{苹果}}=1.0V_{\text{苹果}}
$$

这个结果还要经过后续 Transformer 层和输出层，才会形成预测下一个 Token 的概率分布：

$$
P_\theta(\text{很}\mid\text{苹果})
$$

训练时，用真实目标「很」计算损失。**注意力输出是特征，不等于模型已经直接输出了「很」。**

### 3. 根据「苹果 很」预测「甜」

为什么现在可以融合两个 Token 的信息？因为此时「很」已经属于已知前缀，只有「甜」仍然是下一步要预测的目标。

「很」的 Query 对三个 Key 的分数为 $[1,3,2]$。掩码后：

$$
[1,3,2]\longrightarrow[1,3,-\infty]
$$

再经过 Softmax：

$$
\operatorname{Softmax}([1,3,-\infty])
\approx[0.119,0.881,0]
$$

于是：

$$
z_{\text{很}}=0.119V_{\text{苹果}}+0.881V_{\text{很}}
$$

经过后续网络和输出层，得到：

$$
P_\theta(\text{甜}\mid\text{苹果, 很})
$$

训练时，使用真实目标「甜」计算损失。这一轮的计算只使用「苹果」「很」的信息，没有读取「甜」。

### 4. 为什么训练时能一次处理完整句子？

上面拆成两步，是为了看清模型在预测每个 Token 时实际可以使用什么信息。**GPT 训练时通常采用 Teacher Forcing（教师强制）**：把真实 Token 序列一次送入模型，通过因果掩码同时计算各个 Token 的下一词预测。

例如输入 `[苹果, 很, 甜]`，对应关系为：

| 用哪个 Token 的输出预测 | 该预测可以使用的前缀 | 真实预测目标 |
|:---:|:---|:---:|
| 苹果 | 苹果 | 很 |
| 很 | 苹果、很 | 甜 |
| 甜 | 苹果、很、甜 | 若包含结束标记，则为 `<EOS>` |

这里有一个容易误解的地方：**训练输入中虽然有「很」，但计算「苹果」的输出时，因果掩码不允许它读取「很」的 Key 和 Value。** 所以模型并没有偷看第一项预测的答案。

如果不训练结束标记，本例只计算两个预测目标的损失：

$$
\mathcal L=
-\log P_\theta(\text{很}\mid\text{苹果})
-\log P_\theta(\text{甜}\mid\text{苹果, 很})
$$

这是两项损失之和；实际训练中也常按有效 Token 数量取平均。

### 5. 为什么推理时要逐个生成？

训练时知道完整的真实句子，可以并行计算；推理时只知道用户给出的前缀，**后面的 Token 还没有生成**，因此需要一步一步进行：

1. 输入 `苹果`，预测下一个 Token，例如生成 `很`。
2. 将生成的 `很` 拼接到前缀，得到 `苹果 很`，再预测下一个 Token，例如生成 `甜`。
3. 将 `甜` 拼接到前缀，继续预测，直到生成结束标记或满足停止条件。

生成结果只是举例，实际模型可能生成其他 Token。推理实现中还可以用 KV Cache 缓存历史 Key 和 Value，避免每一步重复计算已有 Token 的注意力键值。

### 6. 和双向自注意力有什么区别？

| 对比 | 双向自注意力 | 因果自注意力 |
|---|---|---|
| Q、K、V 来源 | 同一输入序列 | 同一输入序列 |
| 每个 Token 能读取的信息 | 整句所有 Token | 自身以及此前的 Token |
| 是否使用未来信息掩码 | 否 | 是 |
| 这里的用途 | 融合整句上下文 | 根据已有前缀预测下一个 Token |

两者使用相同的“匹配分数 → Softmax 权重 → 加权融合 Value”计算方法，主要区别是**是否允许读取后续 Token 的信息**。
